# TW Stock HftBacktest Smoke Strategy

Set `SYMBOL`, `START_DATE`, `END_DATE`, `START_TIME`, and `END_TIME` in the next cell. The notebook calls `scripts.tw_stock_data_to_npz.convert_tw_stock_to_npz()` to generate the npz data file before running the strategy.


In [1]:
from dataclasses import replace
from pathlib import Path
import sys

import numpy as np
from numba import njit

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.tw_stock_data_to_npz import convert_tw_stock_to_npz
from scripts.tw_stock_hftbacktest import (
    BacktestConfig,
    build_backtest,
    close_backtest,
    import_hftbacktest,
    wait_for_bbo,
)

SYMBOL = "2330"
START_DATE = "2025-09-09"
END_DATE = START_DATE
START_TIME = None  # e.g. "09:00:00"
END_TIME = None    # e.g. "13:30:00"

DATA_FILE, event_data = convert_tw_stock_to_npz(
    symbol=SYMBOL,
    start_date=START_DATE,
    end_date=END_DATE,
    start_time=START_TIME,
    end_time=END_TIME,
    workspace_root=ROOT,
)

hbtpkg = import_hftbacktest(ROOT)
CONFIG = BacktestConfig(data=DATA_FILE, order_latency_ns=1_000_000)
GTX = hbtpkg.GTX
LIMIT = hbtpkg.LIMIT
DATA_FILE


2026-07-01 10:11:25,049 - INFO - Found 5 parquet files
2026-07-01 10:11:25,050 - INFO - Filtering for date range 20250909 to 20250909


input_rows=46852
converted_rows=46852
skipped_symbol_rows=0
skipped_status_rows=0
skipped_time_rows=0
raw_events=570212
output_events=570212
depth_events=562224
trade_events=7988
opening_jump_qty=2819.0
first_exch_ts=1757377804629985000
last_exch_ts=1757395800000000000
min_feed_latency=0
max_feed_latency=0
qa_rows_checked=1000
best_bid_mismatches=0
best_ask_mismatches=0
trade_qty_mismatches=0
output=C:\Users\zoufuc\Desktop\hftbacktest\data\tw_stock_events\2330_20250909.npz


WindowsPath('C:/Users/zoufuc/Desktop/hftbacktest/data/tw_stock_events/2330_20250909.npz')

In [2]:
@njit
def passive_quote_smoke(hbt, max_steps):
    asset_no = 0
    submitted = False
    steps = 0
    first_bid = 0.0
    first_ask = 0.0

    while steps < max_steps:
        result = hbt.elapse(1_000_000_000)
        if result != 0:
            break

        depth = hbt.depth(asset_no)
        if np.isfinite(depth.best_bid) and np.isfinite(depth.best_ask):
            if first_bid == 0.0:
                first_bid = depth.best_bid
                first_ask = depth.best_ask

            if not submitted:
                bid_px = depth.best_bid - 2.0 * depth.tick_size
                ask_px = depth.best_ask + 2.0 * depth.tick_size
                hbt.submit_buy_order(asset_no, 1, bid_px, 1.0, GTX, LIMIT, False)
                hbt.submit_sell_order(asset_no, 2, ask_px, 1.0, GTX, LIMIT, False)
                submitted = True

        hbt.clear_inactive_orders(asset_no)
        steps += 1

    state = hbt.state_values(asset_no)
    return steps, first_bid, first_ask, hbt.position(asset_no), state.num_trades, state.trading_value


In [3]:
hbt = build_backtest(CONFIG, hbtpkg)
try:
    result = passive_quote_smoke(hbt, 120)
    print("steps, first_bid, first_ask, position, num_trades, trading_value")
    print(result)
finally:
    close_backtest(hbt)


steps, first_bid, first_ask, position, num_trades, trading_value
(120, 1190.0, 1195.0, 0.0, 0, 0.0)


# Bid1/Ask1 Queue Model Fill Wait Test

This test submits one passive GTC limit order at bid1 or ask1, waits for a fill, and compares wait time across queue models.


In [ ]:
QUEUE_MODELS = ("risk_adverse", "power_prob3")
PASSIVE_SIDES = ("buy", "sell")
ENTRY_WARMUP_STEPS = (0, 15, 30)
WAIT_STEP_NS = 1_000_000_000
MAX_WAIT_NS = 120_000_000_000


def wait_one_passive_fill(queue_model, side, warmup_steps, qty=1.0, order_id=20_001):
    config = replace(CONFIG, queue_model=queue_model, order_latency_ns=0)
    hbt = build_backtest(config, hbtpkg)
    asset_no = 0
    try:
        wait_for_bbo(hbt, asset_no)
        for _ in range(warmup_steps):
            if hbt.elapse(WAIT_STEP_NS) != 0:
                break

        depth = hbt.depth(asset_no)
        if side == "buy":
            px = float(depth.best_bid)
            submit_rc = hbt.submit_buy_order(asset_no, order_id, px, qty, hbtpkg.GTC, hbtpkg.LIMIT, False)
        elif side == "sell":
            px = float(depth.best_ask)
            submit_rc = hbt.submit_sell_order(asset_no, order_id, px, qty, hbtpkg.GTC, hbtpkg.LIMIT, False)
        else:
            raise ValueError(side)

        response = hbt.wait_order_response(asset_no, order_id, 10_000_000)
        submit_ts = int(hbt.current_timestamp)
        initial_trades = int(hbt.state_values(asset_no).num_trades)
        waited_ns = 0

        while waited_ns <= MAX_WAIT_NS:
            order = hbt.orders(asset_no).get(order_id)
            state = hbt.state_values(asset_no)
            exec_qty = 0.0 if order is None else float(order.exec_qty)
            leaves_qty = 0.0 if order is None else float(order.leaves_qty)
            filled = order is None or exec_qty > 0.0 or leaves_qty <= 0.0 or int(state.num_trades) > initial_trades
            if filled:
                return {
                    "queue_model": queue_model,
                    "side": side,
                    "warmup_steps": warmup_steps,
                    "price": px,
                    "submit_rc": submit_rc,
                    "response": response,
                    "filled": True,
                    "wait_ns": int(hbt.current_timestamp) - submit_ts,
                    "wait_s": (int(hbt.current_timestamp) - submit_ts) / 1_000_000_000,
                    "exec_qty": exec_qty,
                    "leaves_qty": leaves_qty,
                    "num_trades": int(state.num_trades),
                    "position": float(state.position),
                }

            if hbt.elapse(WAIT_STEP_NS) != 0:
                break
            waited_ns += WAIT_STEP_NS

        order = hbt.orders(asset_no).get(order_id)
        state = hbt.state_values(asset_no)
        return {
            "queue_model": queue_model,
            "side": side,
            "warmup_steps": warmup_steps,
            "price": px,
            "submit_rc": submit_rc,
            "response": response,
            "filled": False,
            "wait_ns": waited_ns,
            "wait_s": waited_ns / 1_000_000_000,
            "exec_qty": 0.0 if order is None else float(order.exec_qty),
            "leaves_qty": 0.0 if order is None else float(order.leaves_qty),
            "num_trades": int(state.num_trades),
            "position": float(state.position),
        }
    finally:
        close_backtest(hbt)


def compare_bid1_ask1_queue_models():
    rows = []
    order_id = 20_001
    for queue_model in QUEUE_MODELS:
        for side in PASSIVE_SIDES:
            for warmup_steps in ENTRY_WARMUP_STEPS:
                rows.append(wait_one_passive_fill(queue_model, side, warmup_steps, order_id=order_id))
                order_id += 1
    return rows


def print_queue_wait_rows(rows):
    header = "model        side warmup filled wait_s price   exec_qty leaves_qty position trades"
    print(header)
    print("-" * len(header))
    for row in rows:
        print(
            f"{row['queue_model']:<12} "
            f"{row['side']:<4} "
            f"{row['warmup_steps']:>6} "
            f"{str(row['filled']):<6} "
            f"{row['wait_s']:>6.1f} "
            f"{row['price']:>7.2f} "
            f"{row['exec_qty']:>8.4f} "
            f"{row['leaves_qty']:>10.4f} "
            f"{row['position']:>8.4f} "
            f"{row['num_trades']:>6}"
        )



def print_queue_wait_diffs(rows, base_model="risk_adverse", compare_model="power_prob3"):
    by_key = {(row["side"], row["warmup_steps"], row["queue_model"]): row for row in rows}
    header = "side warmup base_wait_s prob_wait_s diff_s base_fill prob_fill"
    print(header)
    print("-" * len(header))
    for side in PASSIVE_SIDES:
        for warmup_steps in ENTRY_WARMUP_STEPS:
            base = by_key[(side, warmup_steps, base_model)]
            comp = by_key[(side, warmup_steps, compare_model)]
            diff = comp["wait_s"] - base["wait_s"]
            print(
                f"{side:<4} "
                f"{warmup_steps:>6} "
                f"{base['wait_s']:>11.1f} "
                f"{comp['wait_s']:>11.1f} "
                f"{diff:>6.1f} "
                f"{str(base['filled']):<9} "
                f"{str(comp['filled']):<9}"
            )


In [ ]:
queue_wait_rows = compare_bid1_ask1_queue_models()
print_queue_wait_rows(queue_wait_rows)
print()
print_queue_wait_diffs(queue_wait_rows)
